In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install textstat transformers datasets scikit-learn torch spacy -q
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 4.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.8 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import textstat
import spacy
import pickle
import random
import os
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {0: "elementary", 1: "middle", 2: "high"}
random.seed(42)
np.random.seed(42)

# Load spacy
nlp = spacy.load("en_core_web_sm")
print("✅ spacy loaded")

Using device: cuda
✅ spacy loaded


In [7]:
def compute_static_features(texts):
    features = []
    for text in texts:
        if not text or len(text.strip()) == 0:
            features.append([0.0] * 16)
            continue
        try:
            # Original 10 features
            f = [
                textstat.flesch_reading_ease(text),
                textstat.flesch_kincaid_grade(text),
                textstat.gunning_fog(text),
                textstat.smog_index(text),
                textstat.coleman_liau_index(text),
                textstat.automated_readability_index(text),
                textstat.dale_chall_readability_score(text),
                textstat.avg_sentence_length(text),
                textstat.avg_syllables_per_word(text),
                textstat.lexicon_count(text, removepunct=True) / max(textstat.sentence_count(text), 1)
            ]

            # 6 new domain-agnostic features using spacy
            # truncate to 10000 chars for speed
            doc = nlp(text[:10000])
            tokens = [t for t in doc if not t.is_space]
            words = [t for t in tokens if t.is_alpha]
            sentences = list(doc.sents)

            # Type-token ratio (lexical diversity)
            ttr = len(set([w.lower_ for w in words])) / max(len(words), 1)

            # Named entity density
            ne_density = len(doc.ents) / max(len(sentences), 1)

            # Sentence length variance
            sent_lengths = [len([t for t in s if t.is_alpha]) for s in sentences]
            sent_var = float(np.var(sent_lengths)) if len(sent_lengths) > 1 else 0.0

            # Average dependency distance
            dep_distances = []
            for token in doc:
                if token.head != token:
                    dep_distances.append(abs(token.i - token.head.i))
            avg_dep_dist = float(np.mean(dep_distances)) if dep_distances else 0.0

            # Subordinate clause ratio
            sub_clause_count = sum(1 for t in doc if t.dep_ in ("advcl", "relcl", "ccomp", "xcomp"))
            sub_clause_ratio = sub_clause_count / max(len(sentences), 1)

            # Average word length
            avg_word_len = float(np.mean([len(w.text) for w in words])) if words else 0.0

            f.extend([ttr, ne_density, sent_var, avg_dep_dist, sub_clause_ratio, avg_word_len])

        except Exception as e:
            f = [0.0] * 16

        features.append(f)
    return np.array(features, dtype=np.float32)

print("✅ Static features function defined (16 features)")

✅ Static features function defined (16 features)


In [8]:
print("Loading ScienceQA...")
dataset = load_dataset("nlpscu/Beyond-Flesch", "texts")
scienceqa_train_texts = list(dataset["train"]["full_text"])
scienceqa_train_labels = list(dataset["train"]["education_level"])
scienceqa_test_texts = list(dataset["test"]["full_text"])
scienceqa_test_labels = list(dataset["test"]["education_level"])
print(f"✅ ScienceQA: {len(scienceqa_train_texts)} train, {len(scienceqa_test_texts)} test")

print("Loading OneStopEnglish...")
ose = load_dataset("onestop_english")
ose_texts = list(ose["train"]["text"])
ose_raw_labels = list(ose["train"]["label"])
label_map_ose = {0: "elementary", 1: "middle", 2: "high"}
ose_labels = [label_map_ose[int(l)] for l in ose_raw_labels]
ose_train_texts, ose_test_texts, ose_train_labels, ose_test_labels = train_test_split(
    ose_texts, ose_labels, test_size=0.2, random_state=42, stratify=ose_labels
)
print(f"✅ OSE: {len(ose_train_texts)} train")

Loading ScienceQA...


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/13.8M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3638 [00:00<?, ? examples/s]

...

✅ ScienceQA: 3638 train, 910 test
Loading OneStopEnglish...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/567 [00:00<?, ? examples/s]

✅ OSE: 453 train


In [9]:
TARGET = len(scienceqa_train_texts)  # ~3638 per class

print("Loading RACE...")
race_middle = load_dataset("race", "middle")
race_high = load_dataset("race", "high")

race_mid_texts = list(race_middle["train"]["article"])
race_hi_texts = list(race_high["train"]["article"])
race_test_texts = list(race_middle["test"]["article"]) + list(race_high["test"]["article"])
race_test_labels = ["middle"] * len(race_middle["test"]) + ["high"] * len(race_high["test"])

random.shuffle(race_mid_texts)
random.shuffle(race_hi_texts)
race_mid_sampled = race_mid_texts[:TARGET]
race_hi_sampled = race_hi_texts[:TARGET]
print(f"✅ RACE: {len(race_mid_sampled)} middle, {len(race_hi_sampled)} high")

print("Loading Wikipedia (streaming)...")
simple_wiki = load_dataset("wikimedia/wikipedia", "20231101.simple", streaming=True)
simple_train, simple_test = [], []
for x in simple_wiki["train"]:
    if len(x["text"]) > 200:
        if len(simple_train) < TARGET:
            simple_train.append(x["text"])
        elif len(simple_test) < 300:
            simple_test.append(x["text"])
    if len(simple_train) >= TARGET and len(simple_test) >= 300:
        break

reg_wiki = load_dataset("wikimedia/wikipedia", "20231101.en", streaming=True)
reg_train, reg_test = [], []
for x in reg_wiki["train"]:
    if len(x["text"]) > 200:
        if len(reg_train) < TARGET:
            reg_train.append(x["text"])
        elif len(reg_test) < 300:
            reg_test.append(x["text"])
    if len(reg_train) >= TARGET and len(reg_test) >= 300:
        break

wiki_test_texts = simple_test + reg_test
wiki_test_labels = ["elementary"] * len(simple_test) + ["high"] * len(reg_test)
wiki_test_ids = [label2id[l] for l in wiki_test_labels]
print(f"✅ Wikipedia: {len(simple_train)} simple train, {len(reg_train)} regular train")
print(f"✅ Wikipedia OOD test: {len(wiki_test_texts)}")

Loading RACE...


README.md: 0.00B [00:00, ?B/s]

middle/test-00000-of-00001.parquet:   0%|          | 0.00/405k [00:00<?, ?B/s]

middle/train-00000-of-00001.parquet:   0%|          | 0.00/6.97M [00:00<?, ?B/s]

middle/validation-00000-of-00001.parquet:   0%|          | 0.00/407k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3451 [00:00<?, ? examples/s]

✅ RACE: 3638 middle, 3638 high
Loading Wikipedia (streaming)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

✅ Wikipedia: 3638 simple train, 3638 regular train
✅ Wikipedia OOD test: 600


In [10]:
print("Loading AG News (high/news style)...")
ag_news = load_dataset("ag_news")
ag_texts = list(ag_news["train"]["text"])
random.shuffle(ag_texts)
ag_sampled = ag_texts[:TARGET]
print(f"✅ AG News: {len(ag_sampled)} samples → high label")

print("Loading CBT (elementary)...")
try:
    cbt = load_dataset("cbt", "CN")
    cbt_texts = []
    for x in cbt["train"]:
        # CBT has passages — join them
        passage = " ".join(x["sentences"]) if "sentences" in x else str(x)
        if len(passage) > 100:
            cbt_texts.append(passage)
        if len(cbt_texts) >= TARGET:
            break
    print(f"✅ CBT: {len(cbt_texts)} samples → elementary label")
except Exception as e:
    print(f"❌ CBT failed: {e} — using Simple Wikipedia only for elementary")
    cbt_texts = []

Loading AG News (high/news style)...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

CN/validation-00000-of-00001.parquet:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120769 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ CBT: 3638 samples → elementary label


In [11]:
# Combine all
combined_texts = (
    scienceqa_train_texts +
    ose_train_texts +
    race_mid_sampled +
    race_hi_sampled +
    simple_train +
    reg_train +
    ag_sampled +
    cbt_texts
)

combined_labels = (
    [label2id[l] for l in scienceqa_train_labels] +
    [label2id[l] for l in ose_train_labels] +
    [label2id["middle"]] * len(race_mid_sampled) +
    [label2id["high"]] * len(race_hi_sampled) +
    [label2id["elementary"]] * len(simple_train) +
    [label2id["high"]] * len(reg_train) +
    [label2id["high"]] * len(ag_sampled) +
    [label2id["elementary"]] * len(cbt_texts)
)

scienceqa_test_ids = [label2id[l] for l in scienceqa_test_labels]
race_test_ids = [label2id[l] for l in race_test_labels]

# Check balance
counts = Counter(combined_labels)
print("Class distribution:")
for k, v in counts.items():
    print(f"  {id2label[k]}: {v}")
print(f"Total train: {len(combined_texts)}")

Class distribution:
  high: 12278
  elementary: 8640
  middle: 5001
Total train: 25919


In [12]:
print("Computing train static features (this takes ~20 mins)...")
train_static = compute_static_features(combined_texts)
print("Computing ScienceQA test...")
scienceqa_static = compute_static_features(scienceqa_test_texts)
print("Computing RACE test...")
race_static = compute_static_features(race_test_texts)
print("Computing Wikipedia OOD test...")
wiki_static = compute_static_features(wiki_test_texts)

# Clean NaN
for arr in [train_static, scienceqa_static, race_static, wiki_static]:
    np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0, copy=False)

scaler = StandardScaler()
train_static_scaled = scaler.fit_transform(train_static)
scienceqa_static_scaled = scaler.transform(scienceqa_static)
race_static_scaled = scaler.transform(race_static)
wiki_static_scaled = scaler.transform(wiki_static)
print("✅ Static features done!")

Computing train static features (this takes ~20 mins)...


/tmp/ipykernel_57/265634165.py:17: DeprecationWarning: The 'avg_sentence_length' method has been deprecated due to being the same as 'words_per_sentence'. This method will be removed in thefuture.
  textstat.avg_sentence_length(text),


Computing ScienceQA test...
Computing RACE test...
Computing Wikipedia OOD test...
✅ Static features done!


In [13]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, static_features, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.static = torch.tensor(static_features, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "static": self.static[idx],
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

class DeBERTaWithStatic(nn.Module):
    def __init__(self, model_name, static_dim=16, num_classes=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.static_proj = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 64, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, static_features):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0, :]
        static_out = self.static_proj(static_features)
        combined = torch.cat([cls, static_out], dim=1)
        return self.classifier(combined)

MODEL_NAME = "microsoft/deberta-v3-small"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DeBERTaWithStatic(MODEL_NAME, static_dim=16).to(device)
print("✅ Model ready!")

Loading microsoft/deberta-v3-small...


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

✅ Model ready!


In [14]:
train_dataset = TextDataset(combined_texts, combined_labels, train_static_scaled, tokenizer)
scienceqa_test_dataset = TextDataset(scienceqa_test_texts, scienceqa_test_ids, scienceqa_static_scaled, tokenizer)
race_test_dataset = TextDataset(race_test_texts, race_test_ids, race_static_scaled, tokenizer)
wiki_test_dataset = TextDataset(wiki_test_texts, wiki_test_ids, wiki_static_scaled, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
scienceqa_test_loader = DataLoader(scienceqa_test_dataset, batch_size=32)
race_test_loader = DataLoader(race_test_dataset, batch_size=32)
wiki_test_loader = DataLoader(wiki_test_dataset, batch_size=32)
print(f"✅ Train batches: {len(train_loader)}")

✅ Train batches: 1620


In [15]:
# ── Phase 1: Freeze backbone, train classifier only ───────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Phase 1 — Classifier only (backbone frozen)...")
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

for epoch in range(2):
    model.train()
    total_loss = 0
    for i, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        static = batch["static"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, static)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        if i % 300 == 0:
            print(f"  P1 Epoch {epoch+1} | Step {i}/{len(train_loader)} | Loss: {loss.item():.4f}")
    print(f"Phase1 Epoch {epoch+1} — Avg Loss: {total_loss/len(train_loader):.4f}")

# ── Phase 2: Unfreeze all, fine-tune with fixes ───────────
print("\nPhase 2 — Full fine-tuning (backbone unfrozen)...")
for param in model.encoder.parameters():
    param.requires_grad = True

EPOCHS2 = 3
total_steps = len(train_loader) * EPOCHS2

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-6,
    weight_decay=0.01,
    eps=1e-6
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 5,
    num_training_steps=total_steps
)

for epoch in range(EPOCHS2):
    model.train()
    total_loss = 0
    valid_steps = 0
    for i, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        static = batch["static"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, static)
        loss = criterion(outputs, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        valid_steps += 1

        if i % 300 == 0:
            print(f"  P2 Epoch {epoch+1} | Step {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg = total_loss / max(valid_steps, 1)
    print(f"Phase2 Epoch {epoch+1} — Avg Loss: {avg:.4f} ({valid_steps} valid steps)")

print("\n✅ Training complete!")

Phase 1 — Classifier only (backbone frozen)...
  P1 Epoch 1 | Step 0/1620 | Loss: 1.1199
  P1 Epoch 1 | Step 300/1620 | Loss: 0.8444
  P1 Epoch 1 | Step 600/1620 | Loss: 0.5702
  P1 Epoch 1 | Step 900/1620 | Loss: 0.5670
  P1 Epoch 1 | Step 1200/1620 | Loss: 0.4569
  P1 Epoch 1 | Step 1500/1620 | Loss: 0.4155
Phase1 Epoch 1 — Avg Loss: 0.5736
  P1 Epoch 2 | Step 0/1620 | Loss: 0.5267
  P1 Epoch 2 | Step 300/1620 | Loss: 0.4496
  P1 Epoch 2 | Step 600/1620 | Loss: 0.4634
  P1 Epoch 2 | Step 900/1620 | Loss: 0.3624
  P1 Epoch 2 | Step 1200/1620 | Loss: 0.5027
  P1 Epoch 2 | Step 1500/1620 | Loss: 0.5556
Phase1 Epoch 2 — Avg Loss: 0.5080

Phase 2 — Full fine-tuning (backbone unfrozen)...
  P2 Epoch 1 | Step 0/1620 | Loss: 0.5186
  P2 Epoch 1 | Step 300/1620 | Loss: 0.7927
  P2 Epoch 1 | Step 600/1620 | Loss: 0.8012
  P2 Epoch 1 | Step 900/1620 | Loss: 0.8528
  P2 Epoch 1 | Step 1200/1620 | Loss: 0.5059
  P2 Epoch 1 | Step 1500/1620 | Loss: 0.4980
Phase2 Epoch 1 — Avg Loss: 0.6348 (1620 va

In [16]:
import zipfile
import pickle
import os

save_dir = "/kaggle/working/deberta_static_v3"
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), f"{save_dir}/model_weights.pt")
tokenizer.save_pretrained(save_dir)
with open(f"{save_dir}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Zip for download
zip_path = "/kaggle/working/deberta_static_v3.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in os.listdir(save_dir):
        zipf.write(f"{save_dir}/{file}", file)
size = os.path.getsize(zip_path) / 1024 / 1024
print(f"✅ Saved and zipped: {size:.1f} MB")
print("Download from Output panel NOW!")

✅ Saved and zipped: 250.3 MB
Download from Output panel NOW!


In [20]:
import re
import pandas as pd
import numpy as np

# Load CLEAR corpus
clear_path = "/kaggle/input/datasets/maneeshaprasanna/clear-corpus/CLEAR_corpus_final.xlsx"
print("Loading CLEAR corpus...")
df_clear = pd.read_excel(clear_path)
print(f"Shape: {df_clear.shape}")
print(f"Columns: {df_clear.columns.tolist()}")
print(f"\nLexile Band sample values:")
print(df_clear["Lexile Band"].value_counts().head(20))

# Map Lexile to grade
def lexile_to_grade(val):
    val = str(val).strip()
    numbers = re.findall(r'\d+', val)
    if numbers:
        num = int(numbers[0])
        if num < 600:
            return "elementary"
        elif num < 900:
            return "middle"
        else:
            return "high"
    return None

df_clear["grade_level"] = df_clear["Lexile Band"].apply(lexile_to_grade)
df_clear = df_clear.dropna(subset=["grade_level", "Excerpt"])
df_clear = df_clear[df_clear["Excerpt"].str.len() > 100]

print(f"\nClass distribution after mapping:")
print(df_clear["grade_level"].value_counts())
print(f"\nSample elementary:")
print(df_clear[df_clear["grade_level"]=="elementary"]["Excerpt"].iloc[0][:200])
print(f"\nSample middle:")
print(df_clear[df_clear["grade_level"]=="middle"]["Excerpt"].iloc[0][:200])
print(f"\nSample high:")
print(df_clear[df_clear["grade_level"]=="high"]["Excerpt"].iloc[0][:200])

Loading CLEAR corpus...
Shape: (4724, 28)
Columns: ['ID', 'Author', 'Title', 'Anthology', 'URL', 'Pub Year', 'Categ', 'Sub Cat', 'Lexile Band', 'Location', 'License', 'MPAA Max', 'MPAA #Max', 'MPAA# Avg', 'Excerpt', 'Google WC', 'Sentence Count', 'Paragraphs', 'BT_easiness', 's.e.', 'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level', 'Automated Readability Index', 'SMOG Readability', 'New Dale-Chall Readability Formula', 'CAREC', 'CAREC_M', 'CML2RI']

Lexile Band sample values:
Lexile Band
1100           1379
1300           1359
900             815
700             478
500             364
1500            186
1700             58
300              41
410L-600L        12
610L-800L        12
1900              9
410L - 600L       6
100               3
610L - 800L       1
610L-800          1
Name: count, dtype: int64

Class distribution after mapping:
grade_level
high          3806
middle         492
elementary     426
Name: count, dtype: int64

Sample elementary:
Once upon a time there were 

In [22]:
# Balance CLEAR corpus
print("Balancing CLEAR corpus...")
min_class = df_clear["grade_level"].value_counts().min()
print(f"Min class size: {min_class}")

df_balanced = df_clear.groupby("grade_level").sample(
    n=min_class, random_state=42
).reset_index(drop=True)

print(f"Balanced distribution:")
print(df_balanced["grade_level"].value_counts())

clear_texts = list(df_balanced["Excerpt"])
clear_labels = list(df_balanced["grade_level"])
clear_ids = [label2id[l] for l in clear_labels]

# Compute static features
print("\nComputing CLEAR static features...")
clear_static = compute_static_features(clear_texts)
clear_static = np.nan_to_num(clear_static, nan=0.0, posinf=0.0, neginf=0.0)
clear_static_scaled = scaler.transform(clear_static)

clear_dataset = TextDataset(clear_texts, clear_ids, clear_static_scaled, tokenizer)
clear_loader = DataLoader(clear_dataset, batch_size=32)

# Evaluate all
def evaluate(model, loader, true_ids, label_filter=None):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            static = batch["static"].to(device)
            outputs = model(input_ids, attention_mask, static)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
    pred_labels = [id2label[p] for p in all_preds]
    true_labels = [id2label[t] for t in true_ids]
    if label_filter:
        print(classification_report(true_labels, pred_labels,
              labels=label_filter, zero_division=0))
        return f1_score(true_labels, pred_labels,
               labels=label_filter, average="macro", zero_division=0)
    else:
        print(classification_report(true_labels, pred_labels, zero_division=0))
        return f1_score(true_labels, pred_labels, average="macro", zero_division=0)

print("\n=== ScienceQA Test ===")
scienceqa_f1 = evaluate(model, scienceqa_test_loader, scienceqa_test_ids)
print(f"ScienceQA Macro-F1: {scienceqa_f1:.4f}")

print("\n=== RACE Test ===")
race_f1 = evaluate(model, race_test_loader, race_test_ids,
                   label_filter=["middle", "high"])
print(f"RACE Macro-F1: {race_f1:.4f}")

print("\n=== Wikipedia OOD Test ===")
wiki_f1 = evaluate(model, wiki_test_loader, wiki_test_ids,
                   label_filter=["elementary", "high"])
print(f"Wikipedia OOD Macro-F1: {wiki_f1:.4f}")

print("\n=== CLEAR Corpus OOD (True Unseen — Lexile graded) ===")
clear_f1 = evaluate(model, clear_loader, clear_ids)
print(f"CLEAR Macro-F1: {clear_f1:.4f}")

print("\n=== FINAL SUMMARY ===")
print(f"{'Model':<45} {'ScienceQA':>10} {'RACE':>8} {'Wiki':>8} {'CLEAR(OOD)':>12}")
print(f"{'LR COMBO (Rooein baseline)':<45} {'0.8416':>10} {'N/A':>8} {'N/A':>8} {'N/A':>12}")
print(f"{'BERT+Static v2':<45} {'0.8987':>10} {'0.8049':>8} {'0.9567':>8} {'N/A':>12}")
print(f"{'DeBERTa+Static v3 (8 domains)':<45} {scienceqa_f1:>10.4f} {race_f1:>10.4f} {wiki_f1:>10.4f} {clear_f1:>12.4f}")

Balancing CLEAR corpus...
Min class size: 426
Balanced distribution:
grade_level
elementary    426
high          426
middle        426
Name: count, dtype: int64

Computing CLEAR static features...


/tmp/ipykernel_57/265634165.py:17: DeprecationWarning: The 'avg_sentence_length' method has been deprecated due to being the same as 'words_per_sentence'. This method will be removed in thefuture.
  textstat.avg_sentence_length(text),



=== ScienceQA Test ===
              precision    recall  f1-score   support

  elementary       0.84      0.90      0.87       303
        high       0.80      0.96      0.88       303
      middle       0.82      0.60      0.69       304

    accuracy                           0.82       910
   macro avg       0.82      0.82      0.81       910
weighted avg       0.82      0.82      0.81       910

ScienceQA Macro-F1: 0.8128

=== RACE Test ===
              precision    recall  f1-score   support

      middle       0.59      0.75      0.66      1436
        high       0.88      0.77      0.82      3498

   micro avg       0.77      0.76      0.77      4934
   macro avg       0.74      0.76      0.74      4934
weighted avg       0.80      0.76      0.77      4934

RACE Macro-F1: 0.7404

=== Wikipedia OOD Test ===
              precision    recall  f1-score   support

  elementary       0.96      0.89      0.92       300
        high       0.90      0.96      0.93       300

   micro